# Notebook 67 — Policy Learning

**Policy Learning Specifies Adaptive Controller Candidates**

Notebook 61 converted runtime telemetry into aligned state, action, outcome, and constraint records.
Notebook 67 uses that validated dataset to learn controller candidates before any runtime deployment.

The goal is not to replace the scheduler immediately.
The goal is to learn bounded policy candidates that can be ranked, regularized, and evaluated offline.

In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle

OUT = Path("figures")
OUT.mkdir(exist_ok=True)

plt.rcParams.update({
    "figure.figsize": (14, 8),
    "font.size": 18,
    "axes.titlesize": 34,
    "axes.labelsize": 26,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 18,
})

In [ ]:
def rounded_box(ax, xy, w, h, text, fontsize=18, lw=2.5):
    box = FancyBboxPatch(
        xy, w, h,
        boxstyle="round,pad=0.03,rounding_size=0.08",
        linewidth=lw,
        edgecolor="black",
        facecolor="white",
    )
    ax.add_patch(box)
    ax.text(xy[0] + w/2, xy[1] + h/2, text, ha="center", va="center", fontsize=fontsize)
    return box

def arrow(ax, p1, p2, lw=2.2, mutation_scale=18, connectionstyle="arc3,rad=0"):
    arr = FancyArrowPatch(
        p1, p2,
        arrowstyle="->",
        mutation_scale=mutation_scale,
        linewidth=lw,
        color="black",
        connectionstyle=connectionstyle,
    )
    ax.add_patch(arr)
    return arr

def savefig(name):
    path = OUT / name
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    return path

## 1. Third arc position

In [ ]:
fig, ax = plt.subplots(figsize=(16, 9))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Notebook 67 continues the learning-controller arc", pad=24)

# second arc
second = [
    ("43\nResource\nAllocation", 0.10),
    ("49\nAdaptive\nInfrastructure", 0.31),
    ("53\nDistributed\nScheduling", 0.52),
    ("59\nCluster\nOptimization", 0.73),
]
for label, x in second:
    rounded_box(ax, (x, 0.66), 0.19, 0.14, label, fontsize=16)
for (_, x1), (_, x2) in zip(second[:-1], second[1:]):
    arrow(ax, (x1 + 0.19, 0.73), (x2, 0.73))
ax.text(0.50, 0.58, "second arc: controller → infrastructure → distributed scheduling → cluster optimization",
        ha="center", va="center", fontsize=18)
ax.plot([0.08, 0.92], [0.52, 0.52], color="black", lw=2)

# third arc
third = [
    ("61\nTelemetry\nDataset", 0.08),
    ("67\nPolicy\nLearning", 0.27),
    ("71\nOffline\nEvaluation", 0.46),
    ("73\nSafety\nBounds", 0.65),
    ("79\nAdaptive\nController", 0.84),
]
for label, x in third:
    rounded_box(ax, (x, 0.30), 0.17, 0.14, label, fontsize=16)
for (_, x1), (_, x2) in zip(third[:-1], third[1:]):
    arrow(ax, (x1 + 0.17, 0.37), (x2, 0.37))

# highlight 67
ax.add_patch(Rectangle((0.265, 0.285), 0.18, 0.17, fill=False, lw=4))
arrow(ax, (0.82, 0.66), (0.35, 0.46), connectionstyle="arc3,rad=-0.35")

ax.text(0.50, 0.20, "third arc: telemetry dataset → policy learning → evaluation → safety → controller",
        ha="center", va="center", fontsize=20)
savefig("67_third_arc_position.png")

## 2. Learning pipeline

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Learning dataset specifies policy optimization", pad=24)

labels = [
    "validated\ndataset D",
    "state-action\nrecords",
    "policy\nlearner",
    "candidate\npolicy πθ",
    "offline\nscore",
    "controller\ncandidate",
]
xs = np.linspace(0.07, 0.80, len(labels))
for x, lab in zip(xs, labels):
    rounded_box(ax, (x, 0.45), 0.15, 0.16, lab, fontsize=17)
for x1, x2 in zip(xs[:-1], xs[1:]):
    arrow(ax, (x1 + 0.15, 0.53), (x2, 0.53))

rounded_box(ax, (0.20, 0.16), 0.60, 0.12,
            "Learning produces candidates for offline evaluation, not immediate deployment.",
            fontsize=20)
savefig("67_learning_pipeline.png")

## 3. Policy model inputs

In [ ]:
fig, ax = plt.subplots(figsize=(15, 9))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("State vector maps telemetry into policy candidates", pad=24)

center = rounded_box(ax, (0.39, 0.42), 0.22, 0.15, "policy model\nπθ(S)", fontsize=22)
inputs = [
    ("q(t)\nqueue", (0.08, 0.62)),
    ("u(t)\nutilization", (0.39, 0.74)),
    ("m(t)\nmemory", (0.70, 0.62)),
    ("λ(t)\narrival", (0.08, 0.22)),
    ("r(t)\nreserve", (0.39, 0.10)),
    ("v(t)\nverification", (0.70, 0.22)),
]
for lab, xy in inputs:
    rounded_box(ax, xy, 0.20, 0.12, lab, fontsize=18)
    arrow(ax, (xy[0] + 0.10, xy[1] + 0.06), (0.50, 0.50))

rounded_box(ax, (0.36, 0.23), 0.28, 0.10, "action scores\nπθ(a | S)", fontsize=20)
arrow(ax, (0.50, 0.42), (0.50, 0.33))
ax.text(0.50, 0.04, r"$S(t)=[q(t),u(t),m(t),\lambda(t),r(t),v(t)]$",
        ha="center", fontsize=26)
savefig("67_policy_model_inputs.png")

## 4. Loss objective

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Learning objective balances fit, value, and constraints", pad=24)

parts = [
    ("prediction\nloss", 0.10),
    ("value\nloss", 0.27),
    ("constraint\npenalty", 0.44),
    ("imbalance\npenalty", 0.61),
    ("regularized\nobjective", 0.78),
]
for lab, x in parts:
    rounded_box(ax, (x, 0.50), 0.15, 0.15, lab, fontsize=18)
for x1, x2 in zip([p[1] for p in parts[:-1]], [p[1] for p in parts[1:]]):
    arrow(ax, (x1 + 0.15, 0.575), (x2, 0.575))

eq = r"$\mathcal{L}(\theta)=\ell(\pi_\theta,\pi_0)-\alpha\hat U(\pi_\theta)+\beta C(\pi_\theta)+\gamma R(\pi_\theta)$"
ax.text(0.50, 0.28, eq, ha="center", va="center", fontsize=28)
rounded_box(ax, (0.20, 0.08), 0.60, 0.11,
            "The learner is rewarded for useful policy shifts and penalized for unsafe or unstable ones.",
            fontsize=19)
savefig("67_loss_objective.png")

## 5. Learned policy candidate surface

In [ ]:
reserve = np.linspace(0.05, 0.95, 120)
load = np.linspace(0.05, 0.95, 120)
R, L = np.meshgrid(reserve, load)

# Smooth policy preference proxy:
# 0 steady, 1 reserve, 2 rebalance, 3 scale-out, 4 shed/shorten
score_rebalance = 0.9*L + 0.35*R
score_reserve = 0.85*R - 0.40*L
score_scale = 0.85*R + 0.80*L - 0.85
score_shed = 0.95*L - 0.70*R + 0.10
score_steady = 0.55 - 0.40*np.abs(R-0.30) - 0.35*np.abs(L-0.35)
scores = np.stack([score_steady, score_reserve, score_rebalance, score_scale, score_shed], axis=0)
policy = scores.argmax(axis=0)

fig, ax = plt.subplots(figsize=(13, 10))
im = ax.imshow(policy, origin="lower", extent=[0.05,0.95,0.05,0.95], aspect="auto")
ax.set_title("Learned policy candidate surface", pad=20)
ax.set_xlabel("reserve capacity")
ax.set_ylabel("active load")

regions = [
    ("steady", 0.22, 0.24),
    ("reserve", 0.74, 0.25),
    ("rebalance", 0.52, 0.58),
    ("scale-out", 0.79, 0.83),
    ("shed /\nshorten", 0.20, 0.77),
]
for txt, x, y in regions:
    ax.text(x, y, txt, ha="center", va="center", fontsize=22)

cbar = plt.colorbar(im, ax=ax, fraction=0.05, pad=0.05)
cbar.set_ticks([0,1,2,3,4])
cbar.set_ticklabels(["steady", "reserve", "rebalance", "scale-out", "shed/shorten"])
cbar.set_label("selected candidate action")
savefig("67_policy_candidate_surface.png")

## 6. Counterfactual candidate ranking

In [ ]:
names = ["baseline", "conservative", "balanced", "aggressive", "unsafe"]
throughput = np.array([0.52, 0.57, 0.72, 0.83, 0.92])
latency = np.array([0.25, 0.21, 0.24, 0.38, 0.62])
spillover = np.array([0.18, 0.14, 0.16, 0.25, 0.48])
violation = np.array([0.06, 0.03, 0.05, 0.16, 0.38])
score = throughput - latency - spillover - violation

x = np.arange(len(names))
fig, ax = plt.subplots(figsize=(15, 8))
ax.bar(x - 0.24, throughput, width=0.18, label="gain")
ax.bar(x - 0.06, -latency, width=0.18, label="latency cost")
ax.bar(x + 0.12, -spillover, width=0.18, label="spillover cost")
ax.bar(x + 0.30, score, width=0.18, label="net score")
ax.axhline(0, color="black", lw=1)
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=15)
ax.set_ylabel("offline normalized value")
ax.set_title("Counterfactual ranking selects bounded policy candidates", pad=20)
ax.legend()
savefig("67_counterfactual_ranking.png")

## 7. Generalization gap

In [ ]:
epochs = np.arange(1, 41)
train = 0.82*np.exp(-epochs/18) + 0.08
val = 0.72*np.exp(-epochs/20) + 0.15 + 0.025*np.maximum(epochs-24,0)/16
test = 0.70*np.exp(-epochs/21) + 0.17 + 0.035*np.maximum(epochs-24,0)/16
best = np.argmin(val) + 1

fig, ax = plt.subplots(figsize=(14, 8))
ax.plot(epochs, train, label="train loss")
ax.plot(epochs, val, label="validation loss")
ax.plot(epochs, test, label="test loss")
ax.axvline(best, color="black", linestyle="--", label=f"selected epoch {best}")
ax.set_xlabel("training epoch")
ax.set_ylabel("loss")
ax.set_title("Offline learning needs generalization checks", pad=20)
ax.legend()
savefig("67_generalization_gap.png")

## 8. Constraint regularization

In [ ]:
intensity = np.linspace(0, 1, 200)
gain = 1.10*(1 - np.exp(-3.2*intensity))
latency = 0.18 + 0.75*intensity**3
constraint = 0.10 + 0.95*np.maximum(intensity - 0.63, 0)**2.2
objective_no_penalty = gain - latency
objective_penalty = gain - latency - 1.4*constraint
best = intensity[np.argmax(objective_penalty)]

fig, ax = plt.subplots(figsize=(14, 8))
ax.plot(intensity, gain, label="throughput gain")
ax.plot(intensity, latency, label="latency pressure")
ax.plot(intensity, constraint, label="constraint risk")
ax.plot(intensity, objective_no_penalty, label="unregularized objective")
ax.plot(intensity, objective_penalty, linewidth=3, label="regularized objective")
ax.axvline(best, color="black", linestyle="--", label=f"selected ≈ {best:.2f}")
ax.set_xlabel("policy adaptation intensity")
ax.set_ylabel("normalized value")
ax.set_title("Constraint regularization avoids unsafe adaptation", pad=20)
ax.legend()
savefig("67_constraint_regularization.png")

## 9. Final synthesis

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Policy learning specifies adaptive controller candidates", pad=24)

labels = [
    "validated\ndataset",
    "state\nwindows",
    "outcome\nlabels",
    "policy\nlearner",
    "candidate\nπθ",
    "offline\nranking",
    "bounded\ncontroller",
]
xs = np.linspace(0.03, 0.82, len(labels))
for x, lab in zip(xs, labels):
    rounded_box(ax, (x, 0.46), 0.14, 0.15, lab, fontsize=17)
for x1, x2 in zip(xs[:-1], xs[1:]):
    arrow(ax, (x1 + 0.14, 0.535), (x2, 0.535))

rounded_box(ax, (0.18, 0.18), 0.64, 0.12,
            "Notebook 67 learns candidate policies from telemetry records before runtime control.",
            fontsize=20)
savefig("67_final_synthesis.png")

## Package figures for download

In [ ]:
zip_path = Path("notebook_67_figures_v1.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUT.glob("67_*.png")):
        zf.write(p, arcname=p.name)

manifest = {
    "notebook": "67_policy_learning_v1",
    "title": "Policy Learning Specifies Adaptive Controller Candidates",
    "figures": [p.name for p in sorted(OUT.glob("67_*.png"))],
    "zip": str(zip_path),
}
Path("notebook_67_manifest_v1.json").write_text(json.dumps(manifest, indent=2))

from IPython.display import FileLink, display
display(FileLink(str(zip_path)))
manifest